In [25]:
import sys
sys.path.append("../")

In [26]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [57]:
import torch.nn.functional as F

L = 5
chi = 4
d = 2
qscr = mpsqsc.MpsQsc(L, chi, d, init="stacked", dtype=torch.complex128)
# qscr = qscr.canonicalize(truncate=True)
allup = torch.zeros(L, d, dtype = torch.complex128)
allup[:, 0] = 1.0
alldown = torch.zeros(L, d, dtype = torch.complex128)
alldown[:, 1] = 1.0

allup = mpsqsc.build_product_state(L, d, allup)
alldown = mpsqsc.build_product_state(L, d, alldown)

allup = allup.normalize()
alldown = alldown.normalize()

ghz = mpsqsc.build_ghz_state(L, d, chi)
ghz2 = mpsqsc.build_ghz_state(L, d, chi)
ghz = ghz.normalize()
ghz2 = ghz2.normalize()
As = ghz2.As
As[0][:, 1] = -As[0][:, 1]
ghz2.set_As(As)

In [58]:
qsc = mpsqsc.build_2qsc_from_mpstate(ghz, ghz2)
qsc = qsc.canonicalize(truncate=True)
us, last = qmps.construct_unitary_from_As(qsc.As)

In [60]:
qsc.As[4]

tensor([[[ 0.7071,  0.7071],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [-0.7071,  0.7071]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000]],

        [[ 0.0000,  0.0000],
         [ 0.0000,  0.0000]]], dtype=torch.float64, requires_grad=True)

In [49]:
qmpsqsc = qmps.qMPS(L, 4, d, Us=us, last_unitary=last, seed=42)

ValueError: chi_1=8 must be <= chi=4.

In [34]:
qsc.contract_with_state(ghz), qsc.contract_with_state(allup)

(tensor([1.0000e+00, 9.6606e-18], dtype=torch.float64, grad_fn=<ViewBackward0>),
 tensor([0.7071, 0.7071], dtype=torch.float64, grad_fn=<ViewBackward0>))

In [35]:
qmpsqsc._contract_circuit_with_state_ae(ghz), qmpsqsc._contract_circuit_with_state_ae(allup)

(tensor([[ 1.0000e+00, -1.1102e-16],
         [-5.5511e-17,  6.1630e-33]], dtype=torch.float64,
        grad_fn=<ViewBackward0>),
 tensor([[0.5000, 0.5000],
         [0.5000, 0.5000]], dtype=torch.float64, grad_fn=<ViewBackward0>))

In [6]:
qmpsqsc._build_equation_ae()

('kpab,lqpc,mrqd,nsre,oxs,ktfg,luth,mvui,nwvj,oyw,k,l,m,n,o->xy',
 {'phys_L': ['a', 'b', 'c', 'd', 'e'],
  'phys_R': ['f', 'g', 'h', 'i', 'j'],
  'anc': ['k', 'l', 'm', 'n', 'o'],
  'bond_L': ['p', 'q', 'r', 's'],
  'bond_R': ['t', 'u', 'v', 'w'],
  'cls_L': 'x',
  'cls_R': 'y'},
 <qmpsqsc.models.opt_einsum_utils.GetSymbolFn at 0x153636cd0>)